# Employee Data Cleaning and Visualization

This notebook cleans the employee dataset, exports `cleaned_data.csv`, prints useful code outputs, and creates four visualizations. Run all cells from the repository root.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = Path('sample_data_cleaning_project - Sample_data_cleaning_project.csv')
OUTPUT_FILE = Path('cleaned_data.csv')
PLOT_DIR = Path('visualizations')
PLOT_DIR.mkdir(exist_ok=True)

data = pd.read_csv(INPUT_FILE)
print(f'Raw dataset shape: {data.shape}')
print('Missing values before cleaning:')
print(data.isna().sum())

In [2]:
# Clean missing values and duplicate records.
duplicates_removed = int(data.duplicated().sum())
data = data.drop_duplicates().copy()

for column in data.select_dtypes(include='number').columns:
    data[column] = data[column].fillna(data[column].median())
for column in data.select_dtypes(exclude='number').columns:
    if data[column].isna().any():
        data[column] = data[column].fillna(data[column].mode().iloc[0])

data['Join_Date'] = pd.to_datetime(data['Join_Date'], errors='coerce')
data['Join_Date'] = data['Join_Date'].fillna(data['Join_Date'].min())

# Remove salary outliers with the interquartile-range rule.
q1, q3 = data['Salary'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
before_outliers = len(data)
data = data[data['Salary'].between(lower, upper)].copy()
outliers_removed = before_outliers - len(data)

data = pd.get_dummies(data, columns=['Department'], dtype=int)
data['Join_Date'] = data['Join_Date'].dt.strftime('%Y-%m-%d')
data.to_csv(OUTPUT_FILE, index=False)

print(f'Duplicate rows removed: {duplicates_removed}')
print(f'Salary outliers removed: {outliers_removed}')
print(f'Cleaned dataset shape: {data.shape}')
print('Missing values after cleaning:')
print(data.isna().sum())

In [3]:
print('Cleaned data preview:')
display(data.head())
print('Age and salary summary:')
display(data[['Age', 'Salary']].describe().round(2))

In [4]:
# Reconstruct a readable department label for charts after one-hot encoding.
department_columns = [c for c in data.columns if c.startswith('Department_')]
chart_data = data.copy()
chart_data['Department'] = chart_data[department_columns].idxmax(axis=1).str.replace('Department_', '', regex=False).str.upper()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Employee Dataset Visualizations', fontsize=16, fontweight='bold')

axes[0, 0].hist(chart_data['Age'], bins=8, color='#4C78A8', edgecolor='white')
axes[0, 0].set(title='Age distribution', xlabel='Age', ylabel='Employees')

axes[0, 1].hist(chart_data['Salary'], bins=8, color='#F58518', edgecolor='white')
axes[0, 1].set(title='Salary distribution', xlabel='Salary', ylabel='Employees')

counts = chart_data['Department'].value_counts()
axes[1, 0].bar(counts.index, counts.values, color='#54A24B')
axes[1, 0].set(title='Employees by department', xlabel='Department', ylabel='Employees')

chart_data.boxplot(column='Salary', by='Department', ax=axes[1, 1], grid=False, color='#B279A2')
axes[1, 1].set(title='Salary by department', xlabel='Department', ylabel='Salary')
plt.suptitle('')
plt.tight_layout()
fig.savefig(PLOT_DIR / 'employee_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [5]:
# Save each chart separately for use in the README and visual report.
def save_chart(filename):
    plt.savefig(PLOT_DIR / filename, dpi=150, bbox_inches='tight')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(chart_data['Age'], bins=8, color='#4C78A8', edgecolor='white')
ax.set(title='Age distribution', xlabel='Age', ylabel='Employees')
save_chart('age_distribution.png'); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(chart_data['Salary'], bins=8, color='#F58518', edgecolor='white')
ax.set(title='Salary distribution', xlabel='Salary', ylabel='Employees')
save_chart('salary_distribution.png'); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind='bar', ax=ax, color='#54A24B', title='Employees by department')
ax.set(xlabel='Department', ylabel='Employees')
save_chart('department_counts.png'); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
chart_data.boxplot(column='Salary', by='Department', ax=ax, grid=False)
ax.set(title='Salary by department', xlabel='Department', ylabel='Salary')
plt.suptitle('')
save_chart('salary_by_department.png'); plt.show(); plt.close(fig)

print(f'Saved visualizations to: {PLOT_DIR.resolve()}')